# 02-1 — Prepare all evaluation datasets and cache CLIP features

For each of the paper's evaluation sets this notebook:
1. downloads it with the **authors' own dataset class** (same split, labels, class names, prompt template),
2. generates the **synthetic typographic attack** images with the authors' generator (seed 300, one process per dataset — as the authors ran `eval.py`),
3. encodes clean and attacked images with frozen CLIP ViT-B/32 **exactly as `eval.py` does**, and
4. saves the features to **Google Drive**, then deletes the raw images to free disk.

After this, evaluating any Defense-Prefix vector on all sets takes seconds (`eval_cached.py`).

| Set | Test images | Download | Notes |
|---|---|---|---|
| RTA-100, PAINT | 1000, 110 | in repos | real-world |
| Disentangling (Materzyńska) | ~180 | Google Drive folder | real-world, check file names |
| Pets, Flowers, DTD, EuroSAT | 3.7k, 6.1k, 1.9k, 8.1k | automatic | small |
| Caltech101 | 8.7k | automatic (Google Drive) | fallback to caltech.edu |
| Aircraft, Food101 | 3.3k, 25k | automatic | ~3 GB, ~5 GB |
| ImageNet val | 50k | automatic (authors' downloader) | ~6.4 GB |
| SUN397 | 19.9k | ~37 GB stream | only test files are extracted |
| Stanford Cars | 8k | **done in the Kaggle notebook** | torchvision download is broken |

**Every new session:** run cells 1–4 first, then any dataset cells. Finished datasets are skipped automatically.
**Runtime:** T4 GPU.

In [ ]:
# 1) Google Drive: features and logs are stored here and survive the session
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/dp_project"
CACHE = f"{DRIVE}/cache_test"          # one <dataset>.pt per evaluation set
LOGS  = f"{DRIVE}/logs"
BASE  = "/content"
import os
os.makedirs(CACHE, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
overlay         113G   86M  113G   1% /
Tesla T4, 15360 MiB


In [ ]:
# 2) Dependencies
!pip -q install ftfy regex gdown git+https://github.com/openai/CLIP.git
import sys, torch, PIL, torchvision
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| Pillow", PIL.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
python 3.13.16 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | Pillow 11.3.0


In [ ]:

# Original code, pinned, plus the two compatibility patches from Step 1
import os, pathlib, subprocess
os.chdir(BASE)
if not os.path.isdir("Defense-Prefix"):
    subprocess.run("git clone -q https://github.com/azuma164/Defense-Prefix.git", shell=True, check=True)
os.chdir(f"{BASE}/Defense-Prefix")
subprocess.run("git checkout -q 50d7887", shell=True, check=True)
p = pathlib.Path("datasets/utils/make_dataset_train.py"); s = p.read_text()
if "_textsize" not in s:
    helper = "\ndef _textsize(draw, text, font):\n    # Replacement for ImageDraw.textsize (removed in Pillow 10).\n    l, t, r, b = draw.multiline_textbbox((0, 0), text, font=font)\n    return r, b\n"
    s = s.replace("random.seed(300)", "random.seed(300)\n" + helper, 1).replace("imagedraw.textsize(text, font=font)", "_textsize(imagedraw, text, font)")
    p.write_text(s)
e = pathlib.Path("eval.py"); s = e.read_text()
e.write_text(s.replace("torch.load(args.tokens_path)", "torch.load(args.tokens_path, map_location=device, weights_only=False)"))
print(subprocess.run("git rev-parse HEAD && git diff --stat", shell=True, capture_output=True, text=True).stdout)

50d7887017b295e76b15e56c137ab47e68264448
 datasets/utils/make_dataset_train.py | 10 ++++++++--
 eval.py                              |  2 +-
 2 files changed, 9 insertions(+), 3 deletions(-)



### 3) Our helper scripts (they import the authors' dataset classes and text-encoding function; no method code is changed)

In [ ]:
%%writefile /content/Defense-Prefix/cache_features.py
"""Cache CLIP ViT-B/32 image features for one evaluation dataset of the Defense-Prefix paper.

Run from the root of the Defense-Prefix repository, one dataset per process
(the authors' attack generator seeds `random` once at import, and the authors
ran eval.py once per dataset, so one process per dataset reproduces their setup).

The dataset objects are the authors' own classes from `datasets/`, built with the
same arguments as the authors' `dataset.py`, so splits, labels, class names,
prompt templates and synthetic attacks are exactly theirs. The only difference
from `dataset.py` is that `download` is configurable (Stanford Cars can no longer
be downloaded through torchvision).

Features are computed exactly as in the authors' eval.py: fp16 CLIP image
features, L2-normalised. Evaluating any Defense-Prefix token on these cached
features gives the same predictions as eval.py.
"""
import argparse
import json
import os
import subprocess
import sys
import time

import clip
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.insert(0, os.getcwd())

REAL_WORLD = {"rta-100", "paint", "disentangling"}
device = "cuda" if torch.cuda.is_available() else "cpu"


def build(name, preprocess, root="datasets", download=True):
    """Mirror of the authors' dataset.py with a configurable download flag."""
    if name == "imagenet":
        from datasets.ImageNetV2 import ImageNetValDataset
        return ImageNetValDataset(location=root, transform=preprocess)
    if name == "caltech":
        from datasets.caltech101 import Caltech101
        return Caltech101(root=root, transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "pets":
        from datasets.oxford_pets import OxfordIIITPet
        return OxfordIIITPet(root=root, split="test", transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "cars":
        from datasets.stanford_cars import StanfordCars
        return StanfordCars(root=root, split="test", transform=preprocess, download=False, make_typographic_dataset=True)
    if name == "flowers":
        from datasets.flowers102 import Flowers102
        return Flowers102(root=root, split="test", transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "food":
        from datasets.food101 import Food101
        return Food101(root=root, split="test", transform=preprocess, download=download)
    if name == "aircraft":
        from datasets.fgvcaircraft import FGVCAircraft
        return FGVCAircraft(root=root, split="test", transform=preprocess, download=download)
    if name == "dtd":
        from datasets.dtd import DTD
        return DTD(root=root, split="test", transform=preprocess, download=download)
    if name == "eurosat":
        from datasets.eurosat import EuroSAT
        return EuroSAT(root=root, split="test", transform=preprocess, download=download)
    if name == "sun":
        from datasets.sun397 import SUN397
        return SUN397(root=root, split="test", transform=preprocess, download=False)
    if name == "paint":
        from datasets.paint_ta import PAINTDataset
        return PAINTDataset(root=root, transform=preprocess)
    if name == "disentangling":
        from datasets.disentangling_ta import DisentanglingDataset
        return DisentanglingDataset(root=root, transform=preprocess)
    if name == "rta-100":
        from datasets.rta100 import RTA100
        return RTA100(root=root, transform=preprocess)
    raise ValueError(name)


def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--out", required=True, help="folder for <dataset>.pt")
    ap.add_argument("--root", default="datasets")
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=2)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    model, preprocess = clip.load("ViT-B/32", device=device)
    model.eval()

    t0 = time.time()
    data = build(args.dataset, preprocess, root=args.root)  # also generates the synthetic attack images (seed 300)
    prep_s = time.time() - t0

    loader = DataLoader(data, batch_size=args.batch_size, shuffle=False, num_workers=args.workers)
    real = args.dataset in REAL_WORLD
    clean, attack, labels = [], [], []
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t1 = time.time()
    with torch.no_grad():
        for original_image, typographic_image, target in tqdm(loader, desc=args.dataset):
            f = model.encode_image(original_image.to(device))
            f /= f.norm(dim=-1, keepdim=True)          # identical to eval.py
            clean.append(f.cpu())
            if not real:
                g = model.encode_image(typographic_image.to(device))
                g /= g.norm(dim=-1, keepdim=True)
                attack.append(g.cpu())
            labels.append(torch.as_tensor(target))
    enc_s = time.time() - t1

    out = {
        "dataset": args.dataset,
        "real_world": real,
        "classes": list(data.classes),
        "template": data.templates[0],
        "labels": torch.cat(labels).long(),
        "clean": torch.cat(clean),                       # for real-world sets these images contain the attack
        "attack": torch.cat(attack) if attack else None,
        "meta": {
            "n_images": len(data),
            "n_classes": len(data.classes),
            "prepare_seconds": round(prep_s, 1),
            "encode_seconds": round(enc_s, 1),
            "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if device == "cuda" else None,
            "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
            "torch": torch.__version__,
            "repo_commit": git_commit(),
            "model": "ViT-B/32",
        },
    }
    path = os.path.join(args.out, f"{args.dataset}.pt")
    torch.save(out, path)
    print(json.dumps({"saved": path, **out["meta"]}, indent=1))


if __name__ == "__main__":
    main()


Writing /content/Defense-Prefix/cache_features.py


In [ ]:
%%writefile /content/Defense-Prefix/eval_cached.py
"""Evaluate CLIP and CLIP + Defense-Prefix on cached image features.

Text features and scoring follow the authors' eval.py exactly:
one template per dataset (templates[0]), "* {class}" with the * embedding
replaced by the DP vector, cosine similarity x 100, argmax.

Usage (from the Defense-Prefix repo root):
  python eval_cached.py --cache CACHE_DIR --token learned_token/dp_vit-b32.pt --tag authors_token
"""
import argparse
import csv
import glob
import json
import os
import sys

import clip
import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
from utils.non_nv import encode_text_with_learnt_tokens  # authors' function

device = "cuda" if torch.cuda.is_available() else "cpu"

ORDER = ["imagenet", "caltech", "pets", "cars", "flowers", "food", "aircraft", "dtd", "sun", "eurosat",
         "disentangling", "paint", "rta-100"]
# Paper, Tables B, C and 3 (ViT-B/32): (CLIP, CLIP + DP)
PAPER = {
    "clean": {"imagenet": (62.02, 62.48), "caltech": (88.64, 89.28), "pets": (87.35, 87.22), "cars": (58.72, 57.47),
              "flowers": (66.32, 63.82), "food": (84.14, 83.65), "aircraft": (18.99, 19.26), "dtd": (44.57, 40.64),
              "sun": (61.74, 61.41), "eurosat": (42.98, 43.85)},
    "synthetic": {"imagenet": (39.10, 49.83), "caltech": (63.97, 79.54), "pets": (58.95, 72.88), "cars": (21.02, 28.64),
                  "flowers": (31.32, 44.12), "food": (56.27, 67.79), "aircraft": (10.83, 14.49), "dtd": (25.53, 31.6),
                  "sun": (34.02, 43.50), "eurosat": (4.86, 9.65)},
    "real": {"disentangling": (43.27, 71.93), "paint": (50.00, 63.64), "rta-100": (47.20, 58.00)},
}


def load_model():
    model, _ = clip.load("ViT-B/32", device=device)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    model.eval()
    return model


@torch.no_grad()
def class_text_features(model, classes, template, token=None):
    if token is None:
        toks = torch.cat([clip.tokenize(template.format(c)) for c in classes]).to(device)
        f = model.encode_text(toks)
        return f / f.norm(dim=-1, keepdim=True)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in classes]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    f = model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False)
    return F.normalize(f, dim=-1)


@torch.no_grad()
def predictions(img_feats, text_feats, bs=8192):
    out = []
    for i in range(0, len(img_feats), bs):
        sim = 100.0 * (img_feats[i:i + bs].to(device) @ text_feats.T)
        out.append(sim.argmax(-1).cpu())
    return torch.cat(out)


def evaluate(model, cache_dir, token):
    """Returns rows: dataset, condition, n, acc_clip, acc_dp, plus per-image correctness for later stats."""
    rows, per_image = [], {}
    files = {os.path.basename(p)[:-3]: p for p in glob.glob(os.path.join(cache_dir, "*.pt"))}
    for name in [d for d in ORDER if d in files]:
        c = torch.load(files[name], map_location="cpu", weights_only=False)
        t_clip = class_text_features(model, c["classes"], c["template"])
        t_dp = class_text_features(model, c["classes"], c["template"], token)
        conds = [("real", c["clean"])] if c["real_world"] else [("clean", c["clean"]), ("synthetic", c["attack"])]
        for cond, feats in conds:
            y = c["labels"]
            ok_clip = predictions(feats, t_clip).eq(y)
            ok_dp = predictions(feats, t_dp).eq(y)
            per_image[f"{name}/{cond}"] = {"clip": ok_clip, "dp": ok_dp}
            rows.append({"dataset": name, "condition": cond, "n": len(y),
                         "acc_clip": round(100 * ok_clip.float().mean().item(), 2),
                         "acc_dp": round(100 * ok_dp.float().mean().item(), 2)})
    return rows, per_image


def with_paper(rows):
    for r in rows:
        p = PAPER.get(r["condition"], {}).get(r["dataset"])
        r["paper_clip"], r["paper_dp"] = (p if p else (None, None))
    return rows


def print_table(rows):
    print(f"{'dataset':<14}{'cond':<10}{'n':>7} | {'paper CLIP':>10}{'ours CLIP':>10} | {'paper DP':>9}{'ours DP':>9}")
    for r in rows:
        pc = "" if r["paper_clip"] is None else f"{r['paper_clip']:.2f}"
        pd = "" if r["paper_dp"] is None else f"{r['paper_dp']:.2f}"
        print(f"{r['dataset']:<14}{r['condition']:<10}{r['n']:>7} | {pc:>10}{r['acc_clip']:>10.2f} | {pd:>9}{r['acc_dp']:>9.2f}")
    for cond in ["clean", "synthetic", "real"]:
        sub = [r for r in rows if r["condition"] == cond]
        if sub:
            mc = sum(r["acc_clip"] for r in sub) / len(sub)
            md = sum(r["acc_dp"] for r in sub) / len(sub)
            print(f"  mean {cond:<9} over {len(sub)} sets: CLIP {mc:.2f} | DP {md:.2f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--token", default="learned_token/dp_vit-b32.pt")
    ap.add_argument("--tag", default="authors_token")
    ap.add_argument("--out", default=None, help="results folder (default: <cache>/../results)")
    args = ap.parse_args()

    token = torch.load(args.token, map_location=device, weights_only=False)
    token = token.detach()
    if token.dim() == 1:          # authors save shape (1, 512); eval.py passes token.unsqueeze(0)
        token = token.unsqueeze(0)
    model = load_model()
    token = token.to(device).type(model.dtype)

    rows, per_image = evaluate(model, args.cache, token)
    rows = with_paper(rows)
    print_table(rows)

    out = args.out or os.path.join(os.path.dirname(os.path.abspath(args.cache)), "results")
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, f"{args.tag}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    torch.save(per_image, os.path.join(out, f"{args.tag}_per_image.pt"))
    print("saved", os.path.join(out, f"{args.tag}.csv"))


if __name__ == "__main__":
    main()


Writing /content/Defense-Prefix/eval_cached.py


In [ ]:
# 4) Runner: cache one dataset in its own process, keep the log on Drive, then free disk
import subprocess, shutil, datetime
os.chdir("/content/Defense-Prefix")

def cached(ds):
    return os.path.exists(f"{CACHE}/{ds}.pt")

def run(ds, cleanup=(), extra=""):
    if cached(ds):
        print(f"{ds}: already cached, skipping"); return True
    log = f"{LOGS}/cache_{ds}.log"
    cmd = f"python cache_features.py --dataset {ds} --out {CACHE} --workers 2 {extra} 2>&1 | grep -v -E '^(label=|datasets/|image_|[a-z]/)' | tee {log} | tail -15"
    print(datetime.datetime.now().strftime("%H:%M"), "start", ds)
    subprocess.run(cmd, shell=True)
    ok = cached(ds)
    print(ds, "OK" if ok else "FAILED — see log", log)
    if ok:
        for d in cleanup:
            shutil.rmtree(d, ignore_errors=True)
    !df -h /content | tail -1
    return ok

## Real-world attack sets

In [ ]:
# RTA-100 (inside the authors' repo)
if not cached("rta-100"):
    !unzip -q -o rta100.zip -d /tmp/rta && mkdir -p datasets/ours && rm -rf datasets/ours/data && mv /tmp/rta/rta100 datasets/ours/data
run("rta-100")

rta-100: already cached, skipping


True

In [ ]:
# PAINT real-world set (inside the PAINT repo)
if not cached("paint"):
    !rm -rf /tmp/patching && git clone -q --depth 1 https://github.com/mlfoundations/patching.git /tmp/patching
    !mkdir -p datasets/typographic_real_data_from_PAINT && unzip -q -o /tmp/patching/typographic_attack_data.zip -d datasets/typographic_real_data_from_PAINT
    !ls datasets/typographic_real_data_from_PAINT/typographic_attacks | wc -l   # expect 110
run("paint")

paint: already cached, skipping


True

In [ ]:
# Disentangling (Materzyńska et al.) — via a Drive shortcut (gdown cannot download folders with > 50 files)
import glob, subprocess, zipfile
if not cached("disentangling"):
    root = "datasets/typographic_attack_dataset"
    os.makedirs(root, exist_ok=True)
    def find(name):
        out = subprocess.run(f'find -L /content/drive/MyDrive -maxdepth 5 -name "{name}"',
                             shell=True, capture_output=True, text=True).stdout.split("\n")
        return [p for p in out if p]
    ann, cls = find("annotations.json"), find("classes.json")
    imgs = [p for p in find("images") if os.path.isdir(p)]
    zips = find("*typographic*attack*.zip")
    print("annotations:", ann, "\nclasses:", cls, "\nimages:", imgs, "\nzip:", zips)
    if not (ann and cls and imgs) and zips:          # some copies ship everything inside one zip
        zipfile.ZipFile(zips[0]).extractall("/tmp/disent")
        ann = glob.glob("/tmp/disent/**/annotations.json", recursive=True)
        cls = glob.glob("/tmp/disent/**/classes.json", recursive=True)
        imgs = [d for d in glob.glob("/tmp/disent/**/images", recursive=True) if os.path.isdir(d)]
    if ann and cls and imgs:
        shutil.copy(ann[0], root); shutil.copy(cls[0], root)
        if not os.path.exists(f"{root}/images"): shutil.copytree(imgs[0], f"{root}/images")
        print(len(os.listdir(f"{root}/images")), "images")       # the paper says 171 test images
        run("disentangling")
    else:
        print("Not found. Send me the printed lists above.")

annotations: ['/content/drive/MyDrive/Typographic attack dataset/annotations.json'] 
classes: [] 
images: [] 
zip: []
Not found. Send me the printed lists above.


In [ ]:
# Inspect the Disentangling folder, then build the layout the authors' loader expects
import json, subprocess
SRC = "/content/drive/MyDrive/Typographic attack dataset"
print("== top level")
!ls -la "{SRC}" | head -25
print("== sub-folders")
!find -L "{SRC}" -maxdepth 2 -type d
print("== json files")
!find -L "{SRC}" -maxdepth 3 -name "*.json"

ann = json.load(open(f"{SRC}/annotations.json"))
items = list(ann.items())
print(len(ann), "annotations | example:", items[:2])

# where are the images? look for the first annotated file name anywhere under SRC
first = items[0][0]
hits = subprocess.run(f'find -L "{SRC}" -maxdepth 4 -name "{os.path.basename(first)}"', shell=True,
                      capture_output=True, text=True).stdout.split("\n")
hits = [h for h in hits if h]
img_dir = os.path.dirname(hits[0]) if hits else None
print("image folder:", img_dir)

root = "datasets/typographic_attack_dataset"
os.makedirs(root, exist_ok=True)
cls_files = subprocess.run(f'find -L "{SRC}" -maxdepth 3 -name "classes*.json"', shell=True,
                           capture_output=True, text=True).stdout.split("\n")
cls_files = [c for c in cls_files if c]
if img_dir and items and "true object" in items[0][1]:
    shutil.copy(f"{SRC}/annotations.json", root)
    if cls_files:
        shutil.copy(cls_files[0], f"{root}/classes.json")
        print("classes from", cls_files[0])
    else:
        # DEVIATION: classes.json not in the released folder -> candidates = all true-object + attack labels
        names = sorted({v["true object"] for v in ann.values()} | {v["typographic attack label"] for v in ann.values()})
        json.dump({n: i for i, n in enumerate(names)}, open(f"{root}/classes.json", "w"))
        print("classes.json rebuilt from annotations:", len(names), "names:", names)
    if not os.path.exists(f"{root}/images"):
        shutil.copytree(img_dir, f"{root}/images")
    missing = [k for k in ann if not os.path.exists(f"{root}/images/{k}")]
    print(len(os.listdir(f"{root}/images")), "files copied |", len(missing), "annotated files missing", missing[:5])
    if not missing:
        run("disentangling")
        !python eval_cached.py --cache {CACHE} --token learned_token/dp_vit-b32.pt --tag authors_token | grep -E "dataset|disentangling|real"
else:
    print("Layout still unclear - send me everything printed above.")

== top level
lrw------- 1 root root 0 Nov 12 13:43 /content/drive/MyDrive/Typographic attack dataset -> /content/drive/.shortcut-targets-by-id/17v74pmQs2zHrs9sJ_8D0bfrz5ykB5ZW2/Typographic attack dataset
== sub-folders
/content/drive/MyDrive/Typographic attack dataset
/content/drive/MyDrive/Typographic attack dataset/typographic_attack_dataset
/content/drive/MyDrive/Typographic attack dataset/Pre-trained models
== json files
/content/drive/MyDrive/Typographic attack dataset/annotations.json
180 annotations | example: [('IMG_2934.JPG', {'true object': 'cup', 'typographic attack label': ''}), ('IMG_2935_1.JPG', {'true object': 'cup', 'typographic attack label': 'good'})]
image folder: /content/drive/MyDrive/Typographic attack dataset/typographic_attack_dataset
classes.json rebuilt from annotations: 29 names: ['', 'EU', 'UK', 'backpack', 'bad', 'books', 'cup', 'gloves', 'good', 'hat', 'headphones', 'heater', 'iPad', 'kettle', 'keys', 'money', 'pastries', 'pencil case', 'picture', 'pillow'

In [ ]:
# Disentangling, corrected: drop the empty "" class and the images that carry no attack word
import json
root = "datasets/typographic_attack_dataset"
SRC = "/content/drive/MyDrive/Typographic attack dataset"
ann = json.load(open(f"{SRC}/annotations.json"))
no_attack = [k for k, v in ann.items() if v["typographic attack label"].strip() == ""]
print(len(ann), "images in total |", len(no_attack), "without an attack word:", no_attack[:12])

attacked = {k: v for k, v in ann.items() if v["typographic attack label"].strip() != ""}
names = sorted({v["true object"] for v in attacked.values()} | {v["typographic attack label"] for v in attacked.values()})
objects = sorted({v["true object"] for v in attacked.values()})
words = sorted({v["typographic attack label"] for v in attacked.values()})
print(len(attacked), "attacked images (paper: 171) |", len(objects), "objects |", len(words), "attack words |",
      len(names), "candidate names")

json.dump(attacked, open(f"{root}/annotations.json", "w"))
json.dump({n: i for i, n in enumerate(names)}, open(f"{root}/classes.json", "w"))

# keep the first (180-image) version for the record, out of the way of eval_cached.py
old = f"{CACHE}/disentangling.pt"
if os.path.exists(old):
    os.replace(old, f"{LOGS}/disentangling_all180_with_empty_class.pt.bak")
run("disentangling")
!python eval_cached.py --cache {CACHE} --token learned_token/dp_vit-b32.pt --tag authors_token | grep -E "dataset|disentangling|real"

180 images in total | 20 without an attack word: ['IMG_2934.JPG', 'IMG_2943.JPG', 'IMG_2952.JPG', 'IMG_2961.JPG', 'IMG_2970.JPG', 'IMG_2980.JPG', 'IMG_2989.JPG', 'IMG_3019.JPG', 'IMG_3028.JPG', 'IMG_3037.JPG', 'IMG_3046.JPG', 'IMG_3055.JPG']
160 attacked images (paper: 171) | 20 objects | 8 attack words | 28 candidate names
13:58 start disentangling
disentangling OK /content/drive/MyDrive/dp_project/logs/cache_disentangling.log
overlay         113G 1000M  112G   1% /
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
dataset       cond            n | paper CLIP ours CLIP |  paper DP  ours DP
disentangling real          160 |      43.27     41.87 |     71.93    71.88
paint         real          110 |      50.00     50.00 |     63.64    62.73
rta-100       real         1000 |      47.20     46.90 |     58.00    57.70
  mean real      over 3 sets: CLIP 46.26 | DP 64.10


In [ ]:
# Which single object did the authors leave out? (paper: 171 images = 180 - 9, CLIP 74/171, DP 123/171)
import torch
from eval_cached import load_model, class_text_features, predictions
d = torch.load(f"{LOGS}/disentangling_all180_with_empty_class.pt.bak", map_location="cpu", weights_only=False)
model = load_model()
tok = torch.load("learned_token/dp_vit-b32.pt", map_location="cuda", weights_only=False).detach()
tok = (tok.unsqueeze(0) if tok.dim() == 1 else tok).to("cuda").type(model.dtype)
classes, y, x, tpl = d["classes"], d["labels"], d["clean"], d["template"]
true_names = [classes[i] for i in y.tolist()]
objects = sorted(set(true_names))
print(len(objects), "objects:", objects)

def score(cand, keep):
    yy = torch.tensor([cand.index(n) for n, k in zip(true_names, keep) if k])
    xx = x[torch.tensor(keep)]
    c = int(predictions(xx, class_text_features(model, cand, tpl)).eq(yy).sum())
    p = int(predictions(xx, class_text_features(model, cand, tpl, tok)).eq(yy).sum())
    return len(yy), c, p

print(f"{'left out':<14}{'name in candidates?':<22}{'n':>5}{'CLIP':>6}{'DP':>6}{'CLIP %':>9}{'DP %':>8}  match")
for ex in objects:
    keep = [n != ex for n in true_names]
    for name_kept in (False, True):
        cand = [c for c in classes if c != "" and (name_kept or c != ex)]
        n, c, p = score(cand, keep)
        match = "<== EXACT" if (n, c, p) == (171, 74, 123) else ""
        print(f"{ex:<14}{str(name_kept):<22}{n:>5}{c:>6}{p:>6}{100*c/n:>9.2f}{100*p/n:>8.2f}  {match}")

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 112MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


20 objects: ['backpack', 'books', 'cup', 'gloves', 'hat', 'headphones', 'heater', 'kettle', 'keys', 'pastries', 'pencil case', 'picture', 'pillow', 'pumpkin', 'shoe', 'sunglasses', 'tissues', 'wallet', 'watch', 'watering can']
left out      name in candidates?       n  CLIP    DP   CLIP %    DP %  match
backpack      False                   171    75   122    43.86   71.35  
backpack      True                    171    73   122    42.69   71.35  
books         False                   171    78   125    45.61   73.10  
books         True                    171    78   125    45.61   73.10  
cup           False                   171    78   124    45.61   72.51  
cup           True                    171    78   124    45.61   72.51  
gloves        False                   171    73   122    42.69   71.35  
gloves        True                    171    73   122    42.69   71.35  
hat           False                   171    75   122    43.86   71.35  
hat           True                    

## Synthetic attack sets (clean + attacked versions of each test set)

In [ ]:
# Oxford Pets — also checks that cached evaluation equals the authors' eval.py exactly
if not cached("pets"):
    run("pets")
    !python eval.py --dataset=pets    2>&1 | tail -2
    !python eval.py --dataset=pets -t 2>&1 | tail -2
    !python eval_cached.py --cache {CACHE} --tag check_pets 2>&1 | grep -E "pets|dataset"
    print("The pets numbers from eval.py and eval_cached.py must be identical.")
    shutil.rmtree("datasets/oxford-iiit-pet", ignore_errors=True)

In [ ]:
run("flowers", cleanup=["datasets/flowers-102"])
run("dtd",     cleanup=["datasets/dtd"])
run("eurosat", cleanup=["datasets/eurosat"])

flowers: already cached, skipping
dtd: already cached, skipping
eurosat: already cached, skipping


True

In [ ]:
# Caltech101: torchvision downloads it from Google Drive, which sometimes hits a quota.
# If that fails we fetch the official archive from data.caltech.edu (same images).
if not run("caltech", cleanup=["datasets/caltech101"]):
    !rm -rf datasets/caltech101 && mkdir -p datasets/caltech101
    !wget -q -O /tmp/caltech-101.zip "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip"
    !unzip -q -o /tmp/caltech-101.zip -d /tmp/caltech && tar -xzf /tmp/caltech/caltech-101/101_ObjectCategories.tar.gz -C datasets/caltech101
    !ls datasets/caltech101/101_ObjectCategories | wc -l   # expect 102 (101 classes + background, which torchvision drops)
    run("caltech", cleanup=["datasets/caltech101", "/tmp/caltech", "/tmp/caltech-101.zip"])

caltech: already cached, skipping


In [ ]:
run("aircraft", cleanup=["datasets/fgvc-aircraft-2013b"])

aircraft: already cached, skipping


True

In [ ]:
run("food", cleanup=["datasets/food-101"])   # ~5 GB download

food: already cached, skipping


True

In [ ]:
# ImageNet validation (50k images, 1000 classes) via the authors' own downloader (~6.4 GB)
run("imagenet", cleanup=["datasets/imagenetv2"])

13:24 start imagenet
imagenet FAILED — see log /content/drive/MyDrive/dp_project/logs/cache_imagenet.log
overlay         113G  551M  113G   1% /


False

In [ ]:
# SUN397: the archive is ~37 GB. We stream it and extract only the 19,850 test images of the
# split the authors use (CoOp split, datasets/configs/split_zhou_SUN397.json) plus ClassName.txt.
import json
if not cached("sun"):
    split = json.load(open("datasets/configs/split_zhou_SUN397.json"))
    with open("/tmp/sun_members.txt", "w") as f:
        f.write("SUN397/ClassName.txt\n")
        for path, _, _ in split["test"]:
            f.write(f"SUN397/{path}\n")
    !mkdir -p datasets/sun
    !curl -sL http://vision.princeton.edu/projects/2010/SUN/SUN397.tar.gz | tar -xz -C datasets/sun -T /tmp/sun_members.txt
    !rm -rf datasets/sun/images && mv datasets/sun/SUN397 datasets/sun/images
    !find datasets/sun/images -name "*.jpg" | wc -l    # expect 19850
    run("sun", cleanup=["datasets/sun"])


gzip: stdin: not in gzip format
tar: Child returned status 1
tar: Error is not recoverable: exiting now
mv: cannot stat 'datasets/sun/SUN397': No such file or directory
find: ‘datasets/sun/images’: No such file or directory
0
13:25 start sun
sun FAILED — see log /content/drive/MyDrive/dp_project/logs/cache_sun.log
overlay         113G  553M  113G   1% /


## Check: evaluate the authors' released DP vector on everything cached so far
Our numbers vs the paper (Tables B, C, 3). Stanford Cars appears once the Kaggle notebook's `cars.pt` is copied into `cache_test`.

In [ ]:
!python eval_cached.py --cache {CACHE} --token learned_token/dp_vit-b32.pt --tag authors_token | tee {LOGS}/eval_authors_token.txt
!ls -la {CACHE}

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
dataset       cond            n | paper CLIP ours CLIP |  paper DP  ours DP
caltech       clean        8677 |      88.64     88.71 |     89.28    89.32
caltech       synthetic    8677 |      63.97     64.00 |     79.54    79.61
pets          clean        3669 |      87.35     87.41 |     87.22    87.22
pets          synthetic    3669 |      58.95     59.23 |     72.88    72.42
flowers       clean        6149 |      66.32     66.43 |     63.82    63.83
flowers       synthetic    6149 |      31.32     30.85 |     44.12    44.82
food          clean       25250 |      84.14     84.16 |     83.65    83.62
food          synthetic   25250 |      56.27     57.07 |     67.79    68.91
aircraft      clean        3333 |      18.99     19.17 |     19.26    19.35
aircraft      synthetic    3333 |      10.83     10.71 |     14.49    

In [ ]:
!tail -20 /content/drive/MyDrive/dp_project/logs/cache_imagenet.log
!echo ===== SUN; tail -20 /content/drive/MyDrive/dp_project/logs/cache_sun.log
!echo ===== DIS; tail -20 /content/drive/MyDrive/dp_project/logs/cache_disentangling.log
!ls /content/drive/MyDrive/dp_project/logs

263iB [00:00, 733kiB/s]
Extracting....
Traceback (most recent call last):
  File "/content/Defense-Prefix/cache_features.py", line 147, in <module>
    main()
    ~~~~^^
  File "/content/Defense-Prefix/cache_features.py", line 100, in main
    data = build(args.dataset, preprocess, root=args.root)  # also generates the synthetic attack images (seed 300)
  File "/content/Defense-Prefix/cache_features.py", line 39, in build
    return ImageNetValDataset(location=root, transform=preprocess)
  File "/content/Defense-Prefix/datasets/ImageNetV2.py", line 53, in __init__
    tarfile.open(self.tar_root).extractall(f"{location}")
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/tarfile.py", line 1956, in open
    raise ReadError(f"file could not be opened successfully:\n{error_msgs_summary}")
tarfile.ReadError: file could not be opened successfully:
- method gz: ReadError('not a gzip file')
- method bz2: ReadError('not a bzip2 file')
- method xz: ReadError('not an lzma file')
- metho